# Spam Classifier

A complete machine learning project for text classification using TF-IDF and Logistic Regression.

This notebook demonstrates the full ML workflow: from raw data to a working classifier with error analysis.

## Problem Statement

**Goal:** Build a classifier that accurately distinguishes between spam and legitimate messages.

**Why this problem?**
- Practical real-world application
- Teaches fundamental ML concepts: preprocessing, feature extraction, evaluation
- Good baseline for learning ML fundamentals

**Dataset:** UCI SMS Spam Collection (5,574 messages, 87% legitimate / 13% spam)

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import joblib
import sys
sys.path.append('../src')

# ML Libraries
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

# Visualization
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

print("✓ All imports successful")

## 1. Load and Inspect Dataset

In [ ]:
from data_preprocessing import load_dataset, preprocess_data, get_class_distribution

# Load raw dataset
df = load_dataset('../data/spam.csv')
print(f"Raw dataset: {df.shape}")
print(f"\nData types:\n{df.dtypes}")
print(f"\nFirst messages:\n{df.head()}")

# Check for issues
print(f"\nMissing values: {df.isnull().sum().sum()}")
print(f"Duplicate messages: {df.duplicated(subset=['text']).sum()}")

## 2. Data Cleaning

In [ ]:
# Clean the data
df_clean = preprocess_data(df)
print(f"Cleaned dataset: {df_clean.shape}")
print(f"Rows removed: {len(df) - len(df_clean)}")

# Class distribution
distribution = get_class_distribution(df_clean['label'])
print("\nClass distribution:")
for cls, stats in distribution.items():
    print(f"  {cls}: {stats['count']} ({stats['percentage']}%)")

## 3. Exploratory Data Analysis

In [ ]:
# Message statistics
df_clean['msg_len'] = df_clean['text'].str.len()
df_clean['word_count'] = df_clean['text'].str.split().str.len()

label_names = {0: 'Not Spam', 1: 'Spam'}
df_clean['label_name'] = df_clean['label'].map(label_names)

print("Message statistics by class:")
print(df_clean.groupby('label_name')[['msg_len', 'word_count']].agg(['mean', 'median', 'max']))

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df_clean[df_clean['label']==0]['msg_len'], bins=50, alpha=0.7, label='Not Spam', color='green')
axes[0].hist(df_clean[df_clean['label']==1]['msg_len'], bins=50, alpha=0.7, label='Spam', color='red')
axes[0].set_xlabel('Message Length (characters)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Message Length Distribution')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].hist(df_clean[df_clean['label']==0]['word_count'], bins=50, alpha=0.7, label='Not Spam', color='green')
axes[1].hist(df_clean[df_clean['label']==1]['word_count'], bins=50, alpha=0.7, label='Spam', color='red')
axes[1].set_xlabel('Word Count')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Word Count Distribution')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nObservation: Spam messages tend to be longer than legitimate messages.")

## 4. Train/Test Split

Using stratified split to maintain class balance in both sets (important for imbalanced data).

In [ ]:
# 80/20 split with stratification
X_train, X_test, y_train, y_test = train_test_split(
    df_clean['text'], 
    df_clean['label'],
    test_size=0.2,
    random_state=42,
    stratify=df_clean['label']
)

print(f"Training set: {len(X_train)} messages")
print(f"Test set: {len(X_test)} messages")

print(f"\nTrain distribution:")
print(pd.Series(y_train).value_counts(normalize=True))
print(f"\nTest distribution:")
print(pd.Series(y_test).value_counts(normalize=True))

## 5. Feature Extraction: TF-IDF

**TF-IDF:** Converts text to numerical features by scoring words based on:
- How often they appear in a document (Term Frequency)
- How unique/rare they are across all documents (Inverse Document Frequency)

Words common to spam but rare in legitimate messages get high scores.

In [ ]:
# Create TF-IDF vectorizer
tfidf = TfidfVectorizer(
    max_features=5000,          # Use top 5000 words
    lowercase=True,
    stop_words='english',        # Remove common words
    ngram_range=(1, 2),          # Use single words and bigrams
    min_df=2,                    # Ignore words appearing in < 2 messages
    max_df=0.95                  # Ignore words appearing in > 95% of messages
)

# IMPORTANT: Fit ONLY on training data
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print(f"Features extracted: {X_train_tfidf.shape[1]}")
print(f"Training shape: {X_train_tfidf.shape}")
print(f"Test shape: {X_test_tfidf.shape}")

feature_names = tfidf.get_feature_names_out()
print(f"\nSample features: {list(feature_names[:20])}")

## 6. Train Logistic Regression

**Why Logistic Regression?**
- Interpretable and efficient
- Works well with TF-IDF features
- Probabilistic (gives confidence scores)
- Good baseline for text classification

In [ ]:
# Train model
lr_model = LogisticRegression(
    max_iter=1000,
    random_state=42,
    class_weight='balanced'  # Handle class imbalance
)

print("Training model...")
lr_model.fit(X_train_tfidf, y_train)
print("✓ Training complete")

# Training sanity check
y_train_pred = lr_model.predict(X_train_tfidf)
train_acc = accuracy_score(y_train, y_train_pred)
print(f"Training accuracy: {train_acc:.4f}")

## 7. Model Evaluation

Evaluate on test set using multiple metrics.

**Key metrics for spam classification:**
- **Precision:** Of messages marked as spam, how many actually are? (avoids false alarms)
- **Recall:** Of actual spam, how many did we catch? (avoids missing spam)
- **F1 Score:** Harmonic mean of precision and recall
- **Accuracy:** Overall correctness (can be misleading for imbalanced data)

For spam filters, **False Positives are worse than False Negatives** (losing legitimate emails is bad).

In [ ]:
# Make predictions
y_pred = lr_model.predict(X_test_tfidf)
y_pred_proba = lr_model.predict_proba(X_test_tfidf)

# Calculate metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("=" * 60)
print("EVALUATION METRICS")
print("=" * 60)
print(f"Accuracy:  {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"Precision: {precision:.4f} ({precision*100:.2f}%)")
print(f"Recall:    {recall:.4f} ({recall*100:.2f}%)")
print(f"F1 Score:  {f1:.4f}")

print("\n" + "=" * 60)
print("CLASSIFICATION REPORT")
print("=" * 60)
print(classification_report(y_test, y_pred, target_names=['Not Spam', 'Spam']))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix:")
print(cm)

fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay(cm, display_labels=['Not Spam', 'Spam']).plot(ax=ax, cmap='Blues')
ax.set_title('Confusion Matrix - Spam Classifier')
plt.tight_layout()
plt.show()

print(f"\nTrue Negatives (correct Not Spam):   {cm[0,0]}")
print(f"False Positives (wrongly spam):      {cm[0,1]}")
print(f"False Negatives (missed spam):       {cm[1,0]}")
print(f"True Positives (correct spam):       {cm[1,1]}")

## 8. Error Analysis

Understanding where the model fails is crucial.

In [ ]:
# Find error indices
fp_idx = np.where((y_test == 0) & (y_pred == 1))[0]  # False Positives
fn_idx = np.where((y_test == 1) & (y_pred == 0))[0]  # False Negatives

print(f"FALSE POSITIVES (legitimate marked as spam):")
print(f"Count: {len(fp_idx)} ({len(fp_idx)/len(y_test)*100:.2f}%)")
print("-" * 80)
for i, idx in enumerate(fp_idx[:5]):
    print(f"\n{i+1}. {X_test.iloc[idx]}")
    print(f"   Model probability: {y_pred_proba[idx, 1]:.2%}")

print(f"\n\nFALSE NEGATIVES (spam missed):")
print(f"Count: {len(fn_idx)} ({len(fn_idx)/len(y_test)*100:.2f}%)")
print("-" * 80)
for i, idx in enumerate(fn_idx[:5]):
    print(f"\n{i+1}. {X_test.iloc[idx]}")
    print(f"   Model probability: {y_pred_proba[idx, 1]:.2%}")

print(f"\n\nAnalysis:")
print(f"False Positives are worse → users lose legitimate emails")
print(f"False Negatives are annoying → but users see the spam")

## 9. Example Predictions

In [ ]:
from data_preprocessing import clean_text

test_messages = [
    "You have won a prize! Click here NOW!",
    "Hi, are you free for coffee tomorrow?",
    "URGENT: Your account has been compromised. Call now!",
    "Meeting at 2pm in the conference room",
    "Claim your lottery prize today!",
    "Thanks for the update, I'll see you then",
]

print("EXAMPLE PREDICTIONS")
print("=" * 80)

for msg in test_messages:
    cleaned = clean_text(msg)
    pred = lr_model.predict([cleaned])[0]
    prob = lr_model.predict_proba([cleaned])[0]
    
    label = "SPAM" if pred == 1 else "NOT SPAM"
    confidence = prob[pred]
    
    print(f"\nMessage: {msg}")
    print(f"→ Prediction: {label} (confidence: {confidence:.2%})")
    print("-" * 80)

## 10. Save Model

Save the trained model for use in the Streamlit application.

In [ ]:
# Create models directory
model_dir = Path('../models')
model_dir.mkdir(parents=True, exist_ok=True)

# Save model
model_path = model_dir / 'spam_classifier.joblib'
joblib.dump(lr_model, model_path)
print(f"✓ Model saved: {model_path} ({model_path.stat().st_size / 1024:.1f} KB)")

# Verify
loaded_model = joblib.load(model_path)
test_pred = loaded_model.predict([clean_text("Hello world")])
print(f"✓ Model loaded successfully, test prediction: {test_pred[0]}")